In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import glob
import librosa
import wandb
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics         import f1_score
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from tqdm     import tqdm

BASE        = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUP_DIR  = BASE
ESC_DIR     = f'{BASE}/ESC-50-master/audio'
TEST_CSV    = f'{BASE}/test.csv'
SAMPLE_SUB  = f'{BASE}/sample_submission.csv'
EFF_CACHE   = '/kaggle/working/eff_cache'
AST_CACHE   = '/kaggle/working/ast_cache'
os.makedirs(EFF_CACHE, exist_ok=True)
os.makedirs(AST_CACHE, exist_ok=True)

GENRES   = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
SR       = 22050
SR_AST   = 16000
DURATION = 30
SEG_LEN  = 5
SEG_HOP  = 2
N_MELS   = 128
HOP      = 512
N_MFCC   = 40

le = LabelEncoder()
le.fit(GENRES)

test_df      = pd.read_csv(TEST_CSV)
esc_files    = glob.glob(f'{ESC_DIR}/*.wav')
filename_col = test_df.columns[1]

print(test_df.columns.tolist())
print(test_df.head(3))
print(f'ESC noise files: {len(esc_files)}')
print(f'Sample mashup path: {os.path.join(MASHUP_DIR, test_df.iloc[0][filename_col])}')

In [ ]:
# structure: genres_stems/<genre>/<song_id>/{drums,vocals,bass,others}.wav
STEMS = ['drums', 'vocals', 'bass', 'others']
records = []
for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    for song_id in os.listdir(genre_dir):
        song_dir = os.path.join(genre_dir, song_id)
        if not os.path.isdir(song_dir):
            continue
        for stem in STEMS:
            fp = os.path.join(song_dir, f'{stem}.wav')
            if os.path.exists(fp):
                records.append({'filepath': fp, 'genre': genre, 'song_id': song_id, 'stem': stem})

train_df          = pd.DataFrame(records)
train_df['label'] = le.transform(train_df['genre'])
print(train_df.shape)
print(train_df['genre'].value_counts())

In [ ]:
# trim silence, pad/crop to fixed length, peak-normalise
def preprocess_audio(fp, sr=SR, duration=DURATION, top_db=30):
    y, _ = librosa.load(fp, sr=sr, duration=duration, mono=True)
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals):
        y = np.concatenate([y[s:e] for s, e in intervals])
    target = sr * duration
    if len(y) < target:
        y = np.tile(y, int(np.ceil(target / len(y))))[:target]
    else:
        y = y[:target]
    y = y / (np.max(np.abs(y)) + 1e-9)
    return y

In [ ]:
# mfccs + spectral features flattened to a 1-d vector per clip
def extract_features(fp, sr=SR, n_mfcc=N_MFCC):
    y = preprocess_audio(fp, sr=sr)

    mfcc     = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_d   = librosa.feature.delta(mfcc)
    mfcc_d2  = librosa.feature.delta(mfcc, order=2)
    chroma   = librosa.feature.chroma_stft(y=y, sr=sr)
    mel      = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    tonnetz  = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    zcr      = librosa.feature.zero_crossing_rate(y)
    rms      = librosa.feature.rms(y=y)

    feats = []
    for feat in [mfcc, mfcc_d, mfcc_d2, chroma, mel, contrast, tonnetz, zcr, rms]:
        feats += [feat.mean(axis=1), feat.std(axis=1)]
    return np.concatenate(feats)

In [ ]:
# one feature vector per stem file
X_list, y_list, meta_list = [], [], []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='train features'):
    try:
        X_list.append(extract_features(row['filepath']))
        y_list.append(row['label'])
        meta_list.append((row['genre'], row['song_id'], row['stem']))
    except Exception as e:
        print(f"skip {row['filepath']}: {e}")

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)
print(f'Feature matrix: {X.shape}')

In [ ]:
# test mashups — filename_col already has the relative path from BASE (e.g. mashups/xxxx.wav)
X_test_list = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='test features'):
    fp = os.path.join(MASHUP_DIR, row[filename_col])
    try:
        X_test_list.append(extract_features(fp))
    except:
        X_test_list.append(np.zeros(X.shape[1]))

X_test = np.array(X_test_list, dtype=np.float32)
print(f'Test feature matrix: {X_test.shape}')

In [ ]:
# scale and split — group by song_id so the same song's stems don't bleed into val
from sklearn.model_selection import GroupShuffleSplit

meta_df        = pd.DataFrame(meta_list, columns=['genre','song_id','stem'])
gss            = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=meta_df['song_id']))

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)
X_t_sc = scaler.transform(X_test)

X_tr, X_val = X_sc[train_idx], X_sc[val_idx]
y_tr, y_val = y[train_idx],    y[val_idx]
print(f'Train: {X_tr.shape}  Val: {X_val.shape}')

In [ ]:
# one helper — fit, compute macro f1, log to wandb
wandb.login()

def evaluate_and_log(model, name, config):
    run   = wandb.init(project='DLGENAI', name=name, config=config)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    f1    = f1_score(y_val, preds, average='macro')
    wandb.log({'val_f1_macro': round(f1, 4)})
    print(f'{name:30s}  Macro F1: {f1:.4f}')
    run.finish()
    return f1, model

In [ ]:
# logistic regression
lr = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial', n_jobs=-1)
f1_lr, lr = evaluate_and_log(lr, 'logistic_regression', {'C': 1.0, 'solver': 'lbfgs'})

In [ ]:
# naive bayes
nb = GaussianNB()
f1_nb, nb = evaluate_and_log(nb, 'gaussian_naive_bayes', {'var_smoothing': 1e-9})

In [ ]:
# xgboost
xgb = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', tree_method='gpu_hist', device='cuda',
    random_state=42, n_jobs=-1
)
f1_xgb, xgb = evaluate_and_log(xgb, 'xgboost', {'n_estimators': 500, 'lr': 0.05, 'max_depth': 6})

In [ ]:
# lightgbm
lgbm = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    device='gpu', random_state=42, n_jobs=-1
)
f1_lgbm, lgbm = evaluate_and_log(lgbm, 'lightgbm', {'n_estimators': 500, 'lr': 0.05, 'num_leaves': 63})

In [ ]:
# results summary
results = {'LogReg': f1_lr, 'NaiveBayes': f1_nb, 'XGBoost': f1_xgb, 'LightGBM': f1_lgbm}
print('\n--- Macro F1 Summary ---')
for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {name:15s}: {f1:.4f}')
best_name = max(results, key=results.get)
print(f'\nBest: {best_name} ({results[best_name]:.4f})')